# Healthcare Patient Analytics: Comprehensive EDA Study

## Objective
Perform exploratory analysis on patient healthcare visit data to understand patterns in length of stay, treatment cost, recovery outcomes, and readmission risk. This notebook prepares the dataset for downstream predictive modeling and operational insights.

## Dataset Columns
- `patient_id`: Unique patient identifier
- `visit_date`: Visit timestamp
- `age_group`, `gender`, `region`: Demographics
- `department`, `treatment_type`, `visit_type`: Healthcare service segmentation
- `length_of_stay_days`, `treatment_cost`, `recovery_score`, `readmission_risk`: Outcome and resource metrics
- **Target**: `readmission_risk`

## 1. Environment Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, pearsonr, shapiro, levene
import warnings
warnings.filterwarnings('ignore')

# Visualization configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
%matplotlib inline

# Color palettes
PALETTE = 'viridis'
CATEGORICAL_PALETTE = 'Set2'

print("=" * 60)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 60)
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"SciPy: {scipy.__version__}")

## 2. Data Ingestion & Structural Validation

In [ ]:
# Load dataset
df = pd.read_csv('healthcare_patient_analytics_seaborn.csv')

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Duplicated Rows: {df.duplicated().sum()}")
print(f"\nColumn Data Types:")
print(df.dtypes)
print(f"\nMissing Values (%):")
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0] if missing_pct.sum() > 0 else "No missing values detected.")
print(f"\nFirst 5 Records:")
df.head()

## 3. Data Type Correction & Feature Engineering

In [ ]:
# Parse visit_date column
df['visit_date'] = pd.to_datetime(df['visit_date'], errors='coerce')

# Extract temporal features
df['visit_month'] = df['visit_date'].dt.month
df['day_of_week'] = df['visit_date'].dt.day_name()
df['is_weekend'] = df['visit_date'].dt.dayofweek >= 5
df['quarter'] = df['visit_date'].dt.quarter

# Create proxy numeric age from age groups
df['age'] = df['age_group'].map({
    '<18': 16,
    '18-30': 24,
    '31-45': 38,
    '46-60': 53,
    '60+': 68
})

# Length of stay and cost categories
df['length_of_stay_category'] = pd.cut(df['length_of_stay_days'], 
                                       bins=[-1, 1, 3, 7, df['length_of_stay_days'].max()], 
                                       labels=['Same-day', 'Short', 'Moderate', 'Extended'])
df['cost_category'] = pd.cut(df['treatment_cost'], 
                             bins=[-1, 500, 2000, 5000, df['treatment_cost'].max()], 
                             labels=['Low', 'Medium', 'High', 'Very High'])

df['readmission_risk_category'] = pd.cut(df['readmission_risk'], 
                                         bins=[-0.01, 0.25, 0.5, 0.75, 1.0], 
                                         labels=['Low', 'Moderate', 'High', 'Critical'])

# Recovery categories
df['recovery_category'] = pd.cut(df['recovery_score'], 
                                 bins=[-1, 20, 40, 60, 80, 100], 
                                 labels=['Very Poor', 'Poor', 'Fair', 'Good', 'Excellent'])

print("Feature Engineering Complete.")
print(f"New columns added: {df.shape[1] - 12} engineered features")
print(f"\nEngineered Features Preview:")
df[['age_group', 'age', 'visit_month', 'day_of_week', 'length_of_stay_category', 'cost_category', 'readmission_risk_category', 'recovery_category']].head()

## 4. Comprehensive Descriptive Statistics

In [ ]:
# Numeric columns
numeric_cols = ['age', 'length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

print("=" * 60)
print("DESCRIPTIVE STATISTICS - NUMERIC VARIABLES")
print("=" * 60)
desc_stats = df[numeric_cols].describe().T
desc_stats['skewness'] = df[numeric_cols].skew()
desc_stats['kurtosis'] = df[numeric_cols].kurtosis()
desc_stats['missing'] = df[numeric_cols].isnull().sum()
desc_stats['unique'] = df[numeric_cols].nunique()
print(desc_stats.round(3))

print("\n" + "=" * 60)
print("CATEGORICAL VARIABLES DISTRIBUTION")
print("=" * 60)
cat_cols = ['gender', 'region', 'department', 'treatment_type', 'visit_type', 'length_of_stay_category', 'cost_category', 'readmission_risk_category', 'recovery_category']
for col in cat_cols:
    print(f"\n{col.upper()}:")
    print(df[col].value_counts().head(10))
    print(f"  Unique values: {df[col].nunique()}")

## 5. Univariate Analysis - Distribution Exploration

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

key_vars = ['age', 'length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

for idx, var in enumerate(key_vars):
    sns.histplot(df[var].dropna(), kde=True, ax=axes[idx], color=sns.color_palette(PALETTE, 6)[idx], bins=30)
    axes[idx].set_title(f'Distribution: {var.replace("_", " ").title()}')
    axes[idx].set_xlabel('')
    
    stat, p = shapiro(df[var].dropna().sample(min(5000, len(df[var].dropna()))))
    axes[idx].text(0.95, 0.95, f'Shapiro p={p:.2e}\n{"Normal" if p > 0.05 else "Non-normal"}',
                   transform=axes[idx].transAxes, ha='right', va='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), fontsize=9)

plt.tight_layout()
plt.suptitle('Univariate Distributions with Normality Tests', fontsize=16, y=1.02)
plt.show()

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

outlier_vars = ['length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

for idx, var in enumerate(outlier_vars):
    sns.boxplot(y=df[var], ax=axes[idx], color=sns.color_palette(PALETTE, 4)[idx])
    axes[idx].set_title(f'Outlier Detection: {var.replace("_", " ").title()}')
    
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[var] < Q1 - 1.5*IQR) | (df[var] > Q3 + 1.5*IQR)]
    axes[idx].text(0.5, 0.95, f'Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)',
                   transform=axes[idx].transAxes, ha='center', va='top',
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.suptitle('Outlier Analysis via Box Plots', fontsize=16, y=1.02)
plt.show()

## 6. Categorical Variable Analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

cat_plot_cols = ['gender', 'region', 'department', 'visit_type', 'length_of_stay_category', 'cost_category', 'readmission_risk_category', 'recovery_category', 'quarter']

for idx, col in enumerate(cat_plot_cols):
    if idx >= len(axes):
        break
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[idx], palette=CATEGORICAL_PALETTE)
    axes[idx].set_title(f'Distribution: {col.replace("_", " ").title()}')
    axes[idx].tick_params(axis='x', rotation=45)
    
    total = len(df)
    for p in axes[idx].patches:
        percentage = f'{100 * p.get_height() / total:.1f}%'
        axes[idx].annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                          ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.suptitle('Categorical Variable Distributions', fontsize=16, y=1.02)
plt.show()

## 7. Bivariate Analysis - Target Variable Deep Dive

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

target_col = 'readmission_risk'

sns.histplot(df[target_col], kde=True, ax=axes[0], color='crimson', bins=20)
axes[0].set_title('Target: Readmission Risk Distribution')
axes[0].axvline(df[target_col].mean(), color='black', linestyle='--', label=f'Mean: {df[target_col].mean():.2f}')
axes[0].axvline(df[target_col].median(), color='blue', linestyle='--', label=f'Median: {df[target_col].median():.2f}')
axes[0].legend()

sns.boxplot(y=df[target_col], ax=axes[1], color='lightcoral')
axes[1].set_title('Readmission Risk Box Plot')

sns.violinplot(y=df[target_col], ax=axes[2], color='salmon', inner='quartile')
axes[2].set_title('Readmission Risk Violin Plot')

plt.tight_layout()
plt.show()

print(f"Target Statistics:")
print(f"  Mean: {df[target_col].mean():.3f}")
print(f"  Std: {df[target_col].std():.3f}")
print(f"  Skewness: {df[target_col].skew():.3f}")

print(f"  Kurtosis: {df[target_col].kurtosis():.3f}")
print(f"  Range: [{df[target_col].min()}, {df[target_col].max()}]")

In [ ]:
# Readmission risk by categorical variables
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

target_col = 'readmission_risk'
cat_target_cols = ['gender', 'region', 'department', 'visit_type', 'length_of_stay_category', 'cost_category']

for idx, col in enumerate(cat_target_cols):
    sns.boxplot(data=df, x=col, y=target_col, ax=axes[idx], palette=CATEGORICAL_PALETTE)
    axes[idx].set_title(f'Readmission Risk by {col.replace("_", " ").title()}')
    axes[idx].tick_params(axis='x', rotation=45)
    
    means = df.groupby(col)[target_col].mean()
    for tick, mean_val in enumerate(means):
        axes[idx].plot(tick, mean_val, marker='D', color='red', markersize=8, label='Mean' if tick == 0 else '')

plt.tight_layout()
plt.suptitle('Target Variable Analysis by Categories', fontsize=16, y=1.02)
plt.show()

## 8. Correlation Analysis & Multicollinearity Detection

In [ ]:
# Correlation matrix
corr_vars = ['age', 'length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

corr_matrix = df[corr_vars].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, fmt='.2f', vmin=-1, vmax=1,
            cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Identify high correlations
print("=" * 60)
print("HIGH CORRELATION PAIRS (|r| > 0.5)")
print("=" * 60)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))
            print(f"{corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_val:.3f}")

if not high_corr:
    print("No high correlations (|r| > 0.5) detected. Good for ML modeling.")

# Target-specific correlations
print(f"\nCORRELATIONS WITH TARGET (readmission_risk):")
target_corr = corr_matrix['readmission_risk'].drop('readmission_risk').sort_values(key=abs, ascending=False)
for feat, corr in target_corr.items():
    direction = "↑" if corr > 0 else "↓"
    strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
    print(f"  {direction} {feat}: {corr:.3f} ({strength})")

## 9. Scatter Plot Matrix - Feature Relationships

In [ ]:
key_features = ['age', 'length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

sns.pairplot(df[key_features], diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}, 
             diag_kws={'fill': True}, corner=True, palette=PALETTE)
plt.suptitle('Pairwise Feature Relationships', fontsize=16, y=1.02)
plt.show()

# Detailed scatter: Length of stay and cost vs Readmission Risk
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='length_of_stay_days', y='readmission_risk', 
                hue='gender', alpha=0.6, ax=axes[0], palette='deep')
axes[0].set_title('Length of Stay vs Readmission Risk')

sns.scatterplot(data=df, x='treatment_cost', y='readmission_risk', 
                hue='department', alpha=0.6, ax=axes[1], palette='tab10')
axes[1].set_title('Treatment Cost vs Readmission Risk')

plt.tight_layout()
plt.show()

## 10. Temporal Analysis - Trends Over Time

In [ ]:
# Monthly trends
monthly_trends = df.groupby('visit_month').agg({
    'readmission_risk': 'mean',
    'length_of_stay_days': 'mean',
    'treatment_cost': 'mean',
    'recovery_score': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

metrics = ['readmission_risk', 'length_of_stay_days', 'treatment_cost', 'recovery_score']
colors = ['crimson', 'steelblue', 'green', 'orange']

for idx, metric in enumerate(metrics):
    axes[idx].plot(monthly_trends['visit_month'], monthly_trends[metric], 
                   marker='o', linewidth=2.5, markersize=8, color=colors[idx])
    axes[idx].set_title(f'Monthly Trend: {metric.replace("_", " ").title()}')
    axes[idx].set_xlabel('Month')
    axes[idx].set_xticks(range(1, 13))
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Temporal Trends in Healthcare Metrics', fontsize=16, y=1.02)
plt.show()

# Weekend vs Weekday analysis
weekend_analysis = df.groupby('is_weekend').agg({
    'readmission_risk': ['mean', 'std'],
    'length_of_stay_days': ['mean', 'std'],
    'treatment_cost': ['mean', 'std'],
    'recovery_score': ['mean', 'std']
}).round(2)

print("Weekend vs Weekday Comparison:")
print(weekend_analysis)

## 11. Visit Type & Gender Segmentation Analysis

In [ ]:
# Visit type analysis
platform_analysis = df.groupby('visit_type').agg({
    'readmission_risk': ['mean', 'std', 'count'],
    'length_of_stay_days': 'mean',
    'treatment_cost': 'mean',
    'recovery_score': 'mean'
}).round(2)

print("Visit Type Analysis:")
print(platform_analysis)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(data=df, x='visit_type', y='readmission_risk', ax=axes[0,0], palette='Set1')
axes[0,0].set_title('Readmission Risk by Visit Type')

sns.boxplot(data=df, x='visit_type', y='length_of_stay_days', ax=axes[0,1], palette='Set1')
axes[0,1].set_title('Length of Stay by Visit Type')

sns.boxplot(data=df, x='visit_type', y='treatment_cost', ax=axes[1,0], palette='Set1')
axes[1,0].set_title('Treatment Cost by Visit Type')

sns.boxplot(data=df, x='visit_type', y='recovery_score', ax=axes[1,1], palette='Set1')
axes[1,1].set_title('Recovery Score by Visit Type')

plt.tight_layout()
plt.show()

# Gender and visit type distribution
gender_visit = pd.crosstab(df['gender'], df['visit_type'], normalize='index') * 100
print(f"\nGender-Visit Type Distribution (%):")
print(gender_visit.round(1))

## 12. Lifestyle Interaction Analysis

In [ ]:
# Cost vs Readmission risk interaction
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='length_of_stay_days', y='treatment_cost', 
                hue='readmission_risk', palette='coolwarm', alpha=0.7, ax=axes[0])
axes[0].set_title('Length of Stay vs Treatment Cost (colored by Readmission Risk)')

sns.scatterplot(data=df, x='treatment_cost', y='readmission_risk', 
                hue='recovery_category', palette='viridis', alpha=0.7, ax=axes[1])
axes[1].set_title('Treatment Cost vs Readmission Risk (colored by Recovery Category)')

plt.tight_layout()
plt.show()

# Department interaction analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='length_of_stay_days', y='readmission_risk',
                hue='department', palette='coolwarm', alpha=0.6, ax=axes[0])
axes[0].set_title('Length of Stay vs Readmission Risk by Department')

sns.boxplot(data=df, x='visit_type', y='treatment_cost', ax=axes[1], palette='muted')
axes[1].set_title('Treatment Cost by Visit Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Heatmap: Average readmission risk by recovery and stay category
pivot_risk = df.pivot_table(values='readmission_risk', index='recovery_category', 
                              columns='length_of_stay_category', aggfunc='mean')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_risk, annot=True, cmap='RdYlBu_r', fmt='.2f', linewidths=0.5)
plt.title('Average Readmission Risk: Recovery vs Length of Stay')
plt.show()

## 13. Statistical Hypothesis Testing

In [ ]:
print("=" * 70)
print("STATISTICAL HYPOTHESIS TESTING")
print("=" * 70)

target_col = 'readmission_risk'

# Test 1: Gender differences in readmission risk
male_target = df[df['gender'] == 'Male'][target_col]
female_target = df[df['gender'] == 'Female'][target_col]

levene_stat, levene_p = levene(male_target, female_target)
equal_var = levene_p > 0.05

t_stat, p_val = stats.ttest_ind(male_target, female_target, equal_var=equal_var)
print(f"\nTest 1: Readmission Risk by Gender (Independent t-test)")
print(f"  Male: M={male_target.mean():.2f}, SD={male_target.std():.2f}")
print(f"  Female: M={female_target.mean():.2f}, SD={female_target.std():.2f}")
print(f"  Levene's test p={levene_p:.4f} (equal_var={equal_var})")
print(f"  t={t_stat:.4f}, p={p_val:.4f} → {'Significant' if p_val < 0.05 else 'Not significant'}")

# Test 2: Visit type differences in readmission risk (ANOVA)
platform_groups = [group[target_col].values for name, group in df.groupby('visit_type')]
f_stat, p_val = f_oneway(*platform_groups)
print(f"\nTest 2: Readmission Risk by Visit Type (One-way ANOVA)")
print(f"  F={f_stat:.4f}, p={p_val:.4f} → {'Significant' if p_val < 0.05 else 'Not significant'}")

# Test 3: Department vs Length of Stay Category (Chi-square)
contingency = pd.crosstab(df['department'], df['length_of_stay_category'])
chi2, p_val, dof, expected = chi2_contingency(contingency)
print(f"\nTest 3: Department vs Length of Stay Category (Chi-square)")
print(f"  χ²={chi2:.4f}, p={p_val:.4f}, df={dof}")
print(f"  Cramér's V={np.sqrt(chi2 / (len(df) * (min(contingency.shape) - 1))):.3f}")
print(f"  Result: {'Significant association' if p_val < 0.05 else 'No significant association'}")

# Test 4: Cost correlation with readmission risk
r, p_val = pearsonr(df['treatment_cost'], df[target_col])
print(f"\nTest 4: Treatment Cost vs Readmission Risk (Pearson correlation)")
print(f"  r={r:.4f}, p={p_val:.4e} → {'Significant' if p_val < 0.05 else 'Not significant'}")

# Test 5: Recovery score correlation with readmission risk
r, p_val = pearsonr(df['recovery_score'], df[target_col])
print(f"\nTest 5: Recovery Score vs Readmission Risk (Pearson correlation)")
print(f"  r={r:.4f}, p={p_val:.4e} → {'Significant' if p_val < 0.05 else 'Not significant'}")

## 14. Feature Importance for ML Preparation

In [ ]:
# Mutual information estimation (proxy for feature importance)
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder

# Prepare features
ml_df = df.copy()
le = LabelEncoder()
for col in ['gender', 'region', 'department', 'visit_type', 'age_group', 'length_of_stay_category', 'cost_category', 'recovery_category', 'readmission_risk_category']:
    ml_df[col + '_encoded'] = le.fit_transform(ml_df[col].astype(str))

feature_cols = ['age', 'length_of_stay_days', 'treatment_cost', 'recovery_score',
                'gender_encoded', 'region_encoded', 'department_encoded', 'visit_type_encoded',
                'age_group_encoded', 'length_of_stay_category_encoded', 'cost_category_encoded',
                'recovery_category_encoded', 'readmission_risk_category_encoded']

target_col = 'readmission_risk'
X = ml_df[feature_cols]
y = ml_df[target_col]

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_scores})
mi_df = mi_df.sort_values('MI_Score', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=mi_df, x='MI_Score', y='Feature', palette='viridis')
plt.title('Mutual Information Scores: Feature Importance for Readmission Risk Prediction')
plt.xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print("Top 10 Most Informative Features:")
print(mi_df.head(10).to_string(index=False))

## 15. Outlier Treatment & Data Quality Assessment

In [ ]:
# Z-score based outlier detection
from scipy.stats import zscore

numeric_for_outliers = ['length_of_stay_days', 'treatment_cost', 'recovery_score', 'readmission_risk']

z_scores = np.abs(zscore(df[numeric_for_outliers]))
outlier_mask = (z_scores > 3).any(axis=1)

print(f"Outlier Detection (|z| > 3):")
print(f"  Total outliers: {outlier_mask.sum()} ({outlier_mask.mean()*100:.2f}%)")

# IQR method comparison
iqr_outliers = 0
for col in numeric_for_outliers:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_outliers += ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()

print(f"  IQR method total outlier instances: {iqr_outliers}")

# Distribution before/after log transform for skewed features
skewed_features = ['treatment_cost', 'length_of_stay_days']

fig, axes = plt.subplots(len(skewed_features), 2, figsize=(14, 10))
for idx, feat in enumerate(skewed_features):
    sns.histplot(df[feat], kde=True, ax=axes[idx, 0], color='skyblue')
    axes[idx, 0].set_title(f'Original: {feat}')
    
    log_transformed = np.log1p(df[feat].clip(lower=0))
    sns.histplot(log_transformed, kde=True, ax=axes[idx, 1], color='lightgreen')
    axes[idx, 1].set_title(f'Log-transformed: {feat}')

plt.tight_layout()
plt.suptitle('Skewness Correction via Log Transformation', fontsize=16, y=1.02)
plt.show()

## 16. Risk Segmentation & Cohort Analysis

In [ ]:
# Define risk tiers based on readmission risk
risk_q75 = df['readmission_risk'].quantile(0.75)
risk_q90 = df['readmission_risk'].quantile(0.90)

df['risk_tier'] = pd.cut(df['readmission_risk'], 
                         bins=[-np.inf, df['readmission_risk'].quantile(0.5), risk_q75, risk_q90, np.inf],
                         labels=['Low', 'Moderate', 'High', 'Critical'])

risk_profile = df.groupby('risk_tier').agg({
    'length_of_stay_days': 'mean',
    'treatment_cost': 'mean',
    'recovery_score': 'mean',
    'age': 'mean'
}).round(2)

print("Risk Tier Profiles:")
print(risk_profile)

# Radar chart data preparation
risk_profile_norm = risk_profile.copy()
for col in risk_profile_norm.columns:
    risk_profile_norm[col] = (risk_profile_norm[col] - risk_profile_norm[col].min()) / (risk_profile_norm[col].max() - risk_profile_norm[col].min())

fig, axes = plt.subplots(2, 2, figsize=(16, 14), subplot_kw=dict(polar=True))
axes = axes.flatten()

categories = list(risk_profile_norm.columns)
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

colors = ['green', 'orange', 'red', 'darkred']

for idx, (tier, color) in enumerate(zip(risk_profile_norm.index, colors)):
    values = risk_profile_norm.loc[tier].tolist()
    values += values[:1]
    axes[idx].plot(angles, values, 'o-', linewidth=2, color=color, label=tier)
    axes[idx].fill(angles, values, alpha=0.25, color=color)
    axes[idx].set_xticks(angles[:-1])
    axes[idx].set_xticklabels(categories, size=9)
    axes[idx].set_title(f'{tier} Risk Profile', size=12, pad=20)
    axes[idx].set_ylim(0, 1)

plt.tight_layout()
plt.suptitle('Readmission Risk Tier Radar Profiles', fontsize=16, y=1.02)
plt.show()

print(f"\nRisk Tier Distribution:")
print(df['risk_tier'].value_counts())

## 17. ML Readiness Summary & Recommendations

In [ ]:
print("=" * 70)
print("ML MODEL DEPLOYMENT READINESS REPORT")
print("=" * 70)

# Data quality score
completeness = (1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
uniqueness = (1 - df.duplicated().sum() / len(df)) * 100

print(f"\n📊 DATA QUALITY METRICS:")
print(f"   • Completeness: {completeness:.1f}%")
print(f"   • Uniqueness: {uniqueness:.1f}%")
print(f"   • Total Records: {len(df):,}")
print(f"   • Features Available: {len(feature_cols)}")

print(f"\n🎯 TARGET VARIABLE (readmission_risk):")
print(f"   • Type: {'Continuous' if df['readmission_risk'].nunique() > 10 else 'Discrete'}")
print(f"   • Distribution: {'Normal' if abs(df['readmission_risk'].skew()) < 0.5 else 'Skewed'}")
print(f"   • Range: [{df['readmission_risk'].min()}, {df['readmission_risk'].max()}]")
print(f"   • Recommended Task: {'Regression' if df['readmission_risk'].nunique() > 10 else 'Classification'}")

print(f"\n🔑 TOP PREDICTIVE FEATURES (by correlation & MI):")
top_features = mi_df.head(5)['Feature'].tolist()
for i, feat in enumerate(top_features, 1):
    corr = df[feat].corr(df['readmission_risk']) if feat in df.columns else None
    corr_text = f'{corr:.3f}' if corr is not None else 'N/A'
    print(f"   {i}. {feat} (corr={corr_text})")

print(f"\n⚠️ MULTICOLLINEARITY ALERTS:")
if high_corr:
    for feat1, feat2, corr in high_corr[:3]:
        print(f"   • {feat1} ↔ {feat2}: r={corr:.3f} (consider removing one)")
else:
    print(f"   • No severe multicollinearity detected (all |r| < 0.5)")

print(f"\n📋 PREPROCESSING RECOMMENDATIONS:")
print(f"   1. Encode categoricals: gender, region, department, visit_type → OneHot/Ordinal")
print(f"   2. Scale features: StandardScaler for regression and tree-based models")
print(f"   3. Handle skewness: Log-transform treatment_cost and length_of_stay_days")
print(f"   4. Feature selection: Drop low-MI features (< 0.01)")
print(f"   5. Train/validation split: Stratify by risk_tier if classification approaches are used")

print(f"\n🤖 SUGGESTED MODELS:")
print(f"   • Baseline: Linear Regression / Ridge Regression")
print(f"   • Tree-based: Random Forest, XGBoost, LightGBM")
print(f"   • Neural: MLPRegressor")
print(f"   • Validation: K-Fold Cross-Validation (k=5 or 10)")

print(f"\n💡 KEY BUSINESS INSIGHTS:")
print(f"   • Highest risk group: {df.groupby('risk_tier').size().idxmax()}")
print(f"   • Visit type with highest readmission risk: {df.groupby('visit_type')['readmission_risk'].mean().idxmax()}")
print(f"   • Length of stay and risk correlation: {df['length_of_stay_days'].corr(df['readmission_risk']):.3f}")
print(f"   • Cost and risk correlation: {df['treatment_cost'].corr(df['readmission_risk']):.3f}")
print("=" * 70)